In [26]:
from locallib.picarrodb import *
from locallib.box import *
from locallib.query import *
from locallib.pandas import *

import sqlite3
import os
import pandas as pd

In [27]:
a = Query(get_final_reports('Toscana Energia',years =[2025]))
reports = a.execute([EU1_Conn, EU2_Conn])
print(len(reports))

371


In [28]:
emission_source = f"""SELECT ES.ReportId, ES.Id as EmissionSourceId FROM EmissionSource ES 
    WHERE ES.ReportId IN (SELECT ReportId FROM #TempReports)
    AND (ES.Disposition = 1 OR ES.Disposition =3)"""
reports.db.set_query(emission_source)
emission_source_df = reports.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReports', source_col = 'ReportId', append = False)

18149


In [29]:
box_query = f"""SELECT B.Id as BoxId, 
                B.ReportId, B.UniqueIdentifier
                FROM Box B 
                WHERE B.ReportId IN (SELECT ReportId FROM #TempReports)
                  AND B.UniqueIdentifier NOT LIKE '%G%'"""
           
reports.db.set_query(box_query)
box_df = reports.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReports', source_col = 'ReportId', append = False)

3855


In [30]:
ri_query = f"""SELECT RI.* , 
    (
        SELECT ITT.Name 
        FROM InvestigationTemplateType ITT 
        WHERE ITT.Id = (
            SELECT IT.InvestigationTemplateTypeId 
            FROM InvestigationTemplate IT 
            WHERE IT.Id = RI.InvestigationTemplateId
        )
    ) AS InvestigationTemplateType
    FROM ReportInvestigation RI 
    WHERE RI.BoxId IN (SELECT BoxId FROM #TempBoxes)"""
box_df.db.set_query(ri_query)
ri_df = box_df.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempBoxes', source_col = 'BoxId', append = False)
ri_df.rename(columns = {'Id': 'ReportInvestigationId'}, inplace = True)
ri_df

,ReportInvestigationId,InvestigationTemplateId,LeakFinderUserId,FoundDateTime,LeakLatitude,LeakLongitude,GpsPrecision,Notes,BoxId,CreatedDate,UpdatedDate,UniqueIdentifier,SyncTime,InvestigationTemplateType
0,CD2ED168-BDBD-4227-8790-E67E6612738D,248,6512E2E0-3A0F-F777-DE7B-3A017E5BD096,2025-04-29 08:30:46.730,NaN,NaN,NaN,Dispersione su GRU 10 m3 in linea.,45308F2D-A24C-4D03-8199-0CD656C43BC0,2025-04-29 08:30:46.730,2025-04-29 08:30:46.730,None,NaT,Leak Template
1,2C0FEEF0-6FCD-401D-AA6E-23906B31DE1E,248,37AA07F3-541A-CA47-C6F4-3A18DA5FF393,2025-03-27 10:40:19.627,43.725805,10.767278,3.216000,None,D62BC6F9-8286-4033-9723-02103C021711,2025-03-27 10:40:19.627,2025-03-27 10:40:19.627,None,NaT,Leak Template
2,9F2B4628-4DB8-4FF0-B1AE-F99EACFBD01C,248,49042417-9B65-EAC1-A55D-39FD277F4CB0,2025-02-11 12:48:41.260,NaN,NaN,NaN,None,7B7A23A2-89D0-4C11-9C2E-018E44AB51B0,2025-02-11 12:48:41.260,2025-02-11 12:48:41.260,None,NaT,Leak Template
3,108EE7FB-C8FE-49FD-B3D5-BDC748E57178,248,BA89DF37-6B68-0EDE-5DC0-3A0C756E559C,2025-04-15 10:23:45.340,43.627696,10.294373,4.212000,None,36617E79-C9B3-47DF-9D36-13B6D8155115,2025-04-15 10:23:45.340,2025-04-15 10:23:45.340,None,NaT,Leak Template
4,62E02F89-CB52-467F-AD4F-BEC2EF922488,248,40ABBF4D-3CF9-BE74-7DDB-39F42D5EE194,2025-04-18 13:54:29.660,43.786336,11.229201,5.544328,None,EAE319BF-92CD-4DC3-8CC8-13B7E25FF1D7,2025-04-18 13:54:29.660,2025-04-18 13:54:29.660,None,NaT,Leak Template
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3630,3BFFD2F5-78CE-47E9-BCEC-BA8575BFB717,248,B3D491D2-549F-C4D4-D9B0-39F207122309,2025-09-01 08:33:26.400,43.795398,11.230413,4.745677,None,41C7E215-B359-4F7A-826E-FFE1C99F984E,2025-09-01 08:33:26.400,2025-09-01 08:33:26.400,None,NaT,Leak Template
3631,4AAD5DFC-C9F6-4A44-8F4C-281FF6C1A2DA,669,2073C157-36EF-8587-6F32-3A1414CF5D69,2025-05-19 07:12:58.763,NaN,NaN,NaN,None,28A58551-B120-4F4F-A237-F8F2E521FADF,2025-05-19 07:12:58.763,2025-05-19 07:12:58.763,None,NaT,Other Source Template
3632,94948E54-2703-49DE-8A83-5BFBAF9A5E01,248,217395A6-E155-BC32-8EE0-39F207220067,2025-02-28 08:53:39.037,43.901524,10.918034,2.900000,None,205F455D-0839-41A4-BA11-F8F96B605C7A,2025-02-28 08:53:39.037,2025-02-28 08:53:39.037,None,NaT,Leak Template
3633,64813274-754B-47A4-8FC6-50FD3DAA85B1,248,E210F57D-5C9B-3496-9687-3A12F03BE8D4,2025-03-12 09:52:52.760,43.932220,10.195631,3.505091,None,AB7FBD77-6994-4302-AF0C-F934B2A70F02,2025-03-12 09:52:52.760,2025-03-12 09:52:52.760,None,NaT,Leak Template


In [31]:
rii_query = f""" SELECT RII.ReportInvestigationId, 
                    (SELECT CustomLabel FROM InvestigationTemplateItem ITI WHERE ITI.Id = RII.InvestigationTemplateItemId) AS CustomLabel, 
                    (SELECT Label FROM MasterInvestigationItem WHERE Id = (SELECT MasterInvestigationItemId FROM InvestigationTemplateItem ITI WHERE ITI.Id = RII.InvestigationTemplateItemId)) AS MasterLabel,
                    RII.SelectedValue FROM ReportInvestigationItem RII WHERE RII.ReportInvestigationId IN (SELECT ReportInvestigationId FROM #TempReportInvestigations)"""
ri_df.db.set_query(rii_query)
rii =ri_df.db.execute([EU1_Conn, EU2_Conn],temp_table_name = '#TempReportInvestigations', source_col = 'ReportInvestigationId')

In [32]:
# Pivot ri_df by grouping on 'ReportInvestigationId' with 'CustomLabel' as columns and 'SelectedValue' as values
ri_pivot_df = rii.pivot(index='ReportInvestigationId', columns='MasterLabel', values='SelectedValue')
ri_pivot_df.reset_index(inplace=True) 
ri_pivot_df.head()

MasterLabel,ReportInvestigationId,Città,Commento,Equipment Id,Foto,Grado fuga,Leak Location,Leak Type,LeakType-DropDown,Lettura dopo foro,Lettura in superficie,Materiale tubazione\t Toscana Energia,Matricola Misuratore,Nome strada,Numero di strada,Pratica Sprint Toscana Energia,Reading Unit - Barhole,Reading Unit - Surface,Source,Surface Over Leak
0,00054E1F-86B9-4F82-BC4B-469221E6615A,Pisa,,NaN,,C,Derivazioni parte aeree,Sopra terra,NaN,,1500,Acciaio,0,Via bianchi bandinelli,9,6007,,1,NaN,Piastrellato
1,00392C05-E4F1-478D-8308-43AEA7B63CD4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fognatura,NaN
2,0060D00C-A7FF-4403-9211-3445D37F9DC9,Piombino,,NaN,,C,Derivazioni parte aeree,Sopra terra,NaN,,3200,,0,Via del villaggio dei cavalleggeri,16,19261,,1,NaN,Asfalto
3,006101B5-3D28-4A72-88EA-AA9DBBB0E927,Barberino Tavarnelle,,NaN,,C,Derivazioni parte aeree,Sopra terra,NaN,,2000,Acciaio,Nomadr4015,Via roma,360,39297,,1,NaN,Piastrellato
4,006CF8F5-2A1C-4C54-A124-F6E4584CE97F,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Fognatura,NaN
